In [385]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# Automatically discover candidate mappings for any network

This notebook creates provisional candidate mappings for `NETWORK_ID` from `meta_vars`. It uses variable names, standard names, descriptions, and units to find likely candidates for the six canonical variables.

Automatic discovery is a screening step, not final semantic validation. Review the candidates and their aggregation meaning before marking mappings as accepted.

## 1. Setup

In [386]:
import sys
function_path = './func/'
sys.path.append(function_path)

In [387]:
from IPython.display import display
import sqlalchemy as sa
import pandas as pd
import numpy as np
import re
import os
import sys
from pathlib import Path
from func_variable_mapping import (
    CANONICAL_VARIABLES, CANONICAL_RULES, add_candidate_daily_processing,
    add_candidate_time_scales,
    add_network_station_counts,
    save_candidate_mappings,
)


HERE = Path.cwd()
NETWORK_ID = 76  # Change only this value to work on another network.
NETWORK_DIR = HERE / 'networks' / f'network_{NETWORK_ID}'
NETWORK_DIR.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(HERE / 'func'))
OUTPUT_PREFIX = f'net{NETWORK_ID}_'
DB_URL = os.getenv(
    'CRMP_DB_URL',
    'postgresql://tongli1997@proddb01.pcic.uvic.ca,proddb02.pcic.uvic.ca/crmp'
    '?keepalives=1&keepalives_idle=300&keepalives_interval=300'
    '&keepalives_count=9&passfile=/workspaces/crmprtd/.pgpass',
)
engine = sa.create_engine(DB_URL, pool_pre_ping=True)
MIN_SCORE = 3

## 2. Load the selected network's variable catalog

In [388]:
catalog_query = sa.text("""
SELECT vars_id, network_id, net_var_name::text AS net_var_name, unit,
       standard_name, cell_method, display_name, short_name,
       long_description
FROM meta_vars
WHERE network_id = :network_id
ORDER BY standard_name
""")

network_catalog = pd.read_sql(
    catalog_query, engine, params={'network_id': NETWORK_ID}
)
display(network_catalog)

,vars_id,network_id,net_var_name,unit,standard_name,cell_method,display_name,short_name,long_description
0,1295,76,TEMP_MEAN,celsius,air_temperature,time: mean,Temperature (Mean),air_temperature_mean,Present temperature
1,1297,76,avg_air_temp_pst1hr,celsius,air_temperature,time: mean,Temperature (Mean),T,NaN
2,1298,76,HUMIDITY,%,relative_humidity,time: point,Relative Humidity (Point),relative_humidity_point,Relative humidity
3,1300,76,avg_rel_hum_pst1hr,%,relative_humidity,time: mean,Relative Humidity,rh,NaN
4,1299,76,WDIR_VECT,degree,wind_from_direction,time: point,Wind Direction (Point),wind_from_direction_point,NaN
5,1296,76,WSPD_SCLR,m/s,wind_speed,time: point,Wind Speed (Point),wind_speed_point,NaN


## 3. Define transparent discovery rules

Positive patterns add evidence; exclusion patterns prevent common false matches. Adjust these expressions if the selected network uses unusual abbreviations. The precipitation aggregation is intentionally left pending here: after cadence detection, source metadata classifies it as `as_is`, `sum`, `last`, `difference`, or `fixed_hour_previous_day`.

In [389]:
DISCOVERY_RULES = {
    'air_temperature': {
        'include': [r'air.*temp', r'temp.*air', r'(^|_)temperature($|_)'],
        'exclude': [r'min', r'max', r'dew', r'soil', r'water', r'road', r'surface', r'climatology'],
        'daily_aggregation': 'mean',
    },
    'daily_min_temperature': {
        'include': [r'min(imum)?.*temp', r'temp.*min(imum)?'],
        'exclude': [r'soil', r'water', r'road', r'surface', r'climatology'],
        'daily_aggregation': 'min',
    },
    'daily_max_temperature': {
        'include': [r'max(imum)?.*temp', r'temp.*max(imum)?'],
        'exclude': [r'soil', r'water', r'road', r'surface', r'climatology'],
        'daily_aggregation': 'max',
    },
    'precipitation_amount': {
        'include': [r'precip', r'pcpn', r'(^|_)rain($|_)', r'rainfall'],
        'exclude': [
            r'snow', r'snw', r'climatology', r'detector', r'ratio',
        ],
        'daily_aggregation': 'pending_source_review',
    },
    'snowfall_amount': {
        'include': [r'snow.*fall', r'snwfl', r'standard.*snow'],
        'exclude': [r'depth', r'dpth', r'height', r'climatology'],
        'daily_aggregation': 'sum',
    },
    'snow_depth': {
        'include': [r'snow.*depth', r'depth.*snow', r'snw.*dpth', r'height.*snow'],
        'exclude': [r'fall', r'amount', r'climatology'],
        'daily_aggregation': 'fixed_hour',
    },
}

## 4. Score and rank candidates

In [390]:
def normalized_text(row, columns):
    return ' '.join(str(row.get(column) or '') for column in columns).lower()


def score_candidate(row, canonical_name):
    rule = DISCOVERY_RULES[canonical_name]
    name_text = normalized_text(
        row, ['net_var_name', 'short_name', 'display_name'])
    metadata_text = normalized_text(
        row, ['standard_name', 'long_description', 'cell_method'])
    all_text = name_text + ' ' + metadata_text

    if any(re.search(pattern, all_text) for pattern in rule['exclude']):
        return 0, 'excluded keyword'

    name_hits = sum(bool(re.search(pattern, name_text))
                    for pattern in rule['include'])
    metadata_hits = sum(bool(re.search(pattern, metadata_text))
                        for pattern in rule['include'])
    score = 3 * name_hits + 3 * metadata_hits

    expected_unit = CANONICAL_RULES[canonical_name].unit.lower()
    actual_unit = str(row.get('unit') or '').lower()
    unit_tokens = {
        'celsius': ['celsius', 'degc', 'degree_c', 'degrees_c', '°c'],
        'mm': ['mm', 'millimet'],
        'cm': ['cm', 'centimet'],
    }[expected_unit]
    unit_match = any(token in actual_unit for token in unit_tokens)
    if unit_match:
        score += 1

    evidence = f'name hits={name_hits}; metadata hits={metadata_hits}; unit match={unit_match}'
    return score, evidence


rows = []
for _, variable in network_catalog.iterrows():
    for canonical_name in CANONICAL_VARIABLES:
        score, evidence = score_candidate(variable, canonical_name)
        if score >= MIN_SCORE:
            rule = DISCOVERY_RULES[canonical_name]
            rows.append({
                'vars_id': int(variable['vars_id']),
                'net_var_name': variable['net_var_name'],
                'canonical_variable': canonical_name,
                'unit': variable['unit'],
                'daily_aggregation': rule['daily_aggregation'],
                'source_family': f'network{NETWORK_ID}_auto',
                'discovery_score': score,
                'discovery_evidence': evidence,
                'mapping_status': 'needs_review',
            })

candidate_columns = [
    'vars_id', 'net_var_name', 'canonical_variable', 'unit',
    'daily_aggregation', 'source_family', 'discovery_score',
    'discovery_evidence', 'mapping_status', 'priority',
]
discovered_candidates = pd.DataFrame(rows, columns=candidate_columns)
if not discovered_candidates.empty:
    discovered_candidates = discovered_candidates.sort_values(
        ['canonical_variable', 'discovery_score', 'vars_id'],
        ascending=[True, False, True],
    )
    discovered_candidates['priority'] = (
        discovered_candidates.groupby('canonical_variable').cumcount() + 1
    )
display(discovered_candidates)

,vars_id,net_var_name,canonical_variable,unit,daily_aggregation,source_family,discovery_score,discovery_evidence,mapping_status,priority
0,1295,TEMP_MEAN,air_temperature,celsius,mean,network76_auto,13,name hits=3; metadata hits=1; unit match=True,needs_review,1
1,1297,avg_air_temp_pst1hr,air_temperature,celsius,mean,network76_auto,7,name hits=1; metadata hits=1; unit match=True,needs_review,2


## 5. Review and override

List false-positive IDs under `EXCLUDE`. Use `MANUAL_CANDIDATES` for variables missed by the patterns or to override inferred semantics. Re-run this cell after editing the lists.

In [391]:
# Persist reviewed overrides by network so changing NETWORK_ID is safe.
EXCLUDE_BY_NETWORK = {
    19: {(837, 'precipitation_amount')},  # Candidate, but excluded.
    6: {
        (1204, 'precipitation_amount'),
        (1207, 'precipitation_amount'),
        (1211, 'precipitation_amount'), },  # Candidates, but excluded.
    11: {(1268, 'air_temperature')},  # Candidate, but excluded.
    36: {(737, 'precipitation_amount')},  # Candidate, but excluded.

}
MANUAL_CANDIDATES_BY_NETWORK = {
    1: {432: 'snow_depth'},  # SNOW_ON_THE_GROUND is depth, not snowfall.
    5: {468: 'snow_depth'},  # SNOW_ON_THE_GROUND is depth, not snowfall.
}
# Only nonzero-station candidates can need a reviewed cadence override.
SOURCE_TIME_SCALE_OVERRIDES_BY_NETWORK = {}
EXCLUDE = EXCLUDE_BY_NETWORK.get(NETWORK_ID, set())
MANUAL_CANDIDATES = MANUAL_CANDIDATES_BY_NETWORK.get(NETWORK_ID, {})
SOURCE_TIME_SCALE_OVERRIDES = SOURCE_TIME_SCALE_OVERRIDES_BY_NETWORK.get(
    NETWORK_ID, {})

reviewed = discovered_candidates.copy()
if not reviewed.empty and EXCLUDE:
    excluded = pd.MultiIndex.from_tuples(EXCLUDE)
    row_keys = pd.MultiIndex.from_frame(
        reviewed[['vars_id', 'canonical_variable']])
    reviewed.loc[row_keys.isin(excluded), 'mapping_status'] = 'excluded'

if MANUAL_CANDIDATES:
    manual_rows = []
    for vars_id, canonical_name in MANUAL_CANDIDATES.items():
        if canonical_name not in CANONICAL_VARIABLES:
            raise ValueError(
                f'Unknown canonical variable for vars_id {vars_id}: {canonical_name}'
            )
        catalog_match = network_catalog[network_catalog['vars_id'] == vars_id]
        if catalog_match.empty:
            raise ValueError(
                f'vars_id {vars_id} is not in network {NETWORK_ID}')
        variable = catalog_match.iloc[0]
        manual_rows.append({
            'vars_id': int(vars_id),
            'net_var_name': variable['net_var_name'],
            'canonical_variable': canonical_name,
            'unit': variable['unit'],
            'daily_aggregation': DISCOVERY_RULES[canonical_name]['daily_aggregation'],
            'source_family': f'network{NETWORK_ID}_reviewed',
            'discovery_score': np.nan,
            'discovery_evidence': 'manual reviewed mapping',
            'mapping_status': 'candidate',
        })

    manual = pd.DataFrame(manual_rows)
    if not reviewed.empty:
        reviewed = reviewed.loc[
            ~reviewed['vars_id'].isin(MANUAL_CANDIDATES)
        ].copy()
    reviewed = pd.concat([reviewed, manual], ignore_index=True)

# Manual candidates sort first and excluded candidates last.
reviewed['_status_order'] = reviewed['mapping_status'].map(
    {'candidate': 0, 'needs_review': 1, 'excluded': 2}
).fillna(1)
reviewed = reviewed.sort_values(
    ['canonical_variable', '_status_order', 'discovery_score', 'vars_id'],
    ascending=[True, True, False, True],
).reset_index(drop=True)
reviewed['priority'] = reviewed.groupby('canonical_variable').cumcount() + 1
reviewed = reviewed.drop(columns='_status_order')
display(reviewed)

,vars_id,net_var_name,canonical_variable,unit,daily_aggregation,source_family,discovery_score,discovery_evidence,mapping_status,priority
0,1295,TEMP_MEAN,air_temperature,celsius,mean,network76_auto,13,name hits=3; metadata hits=1; unit match=True,needs_review,1
1,1297,avg_air_temp_pst1hr,air_temperature,celsius,mean,network76_auto,7,name hits=1; metadata hits=1; unit match=True,needs_review,2


## 6. Build and save this network's candidate mappings

The diagnostic columns remain in `discovered_candidates`; the final registry is saved as `networks/network_<id>/net<id>_candidate_mappings.csv`. `network_station_count` is the number of distinct stations in the network with at least one non-null observation for that candidate at any date.

In [392]:
mapping_columns = [
    'vars_id', 'net_var_name', 'canonical_variable',
    'daily_aggregation', 'priority', 'source_family', 'mapping_status',
]
candidate_mappings = (
    reviewed[mapping_columns]
    .sort_values(['canonical_variable', 'priority', 'vars_id'])
    .reset_index(drop=True)
)
candidate_mappings = add_network_station_counts(
    engine, candidate_mappings, network_id=NETWORK_ID)
candidate_mappings = add_candidate_time_scales(
    engine, candidate_mappings, network_catalog, network_id=NETWORK_ID,
    overrides=SOURCE_TIME_SCALE_OVERRIDES)
# Daily source values are already daily summaries and therefore use as_is.
# Hourly sources retain the aggregation appropriate to the variable.
candidate_mappings = add_candidate_daily_processing(
    candidate_mappings, network_catalog)
# display(candidate_mappings)

# Save or replace this network in the prefixed candidate-mappings CSV.
all_saved_mappings = save_candidate_mappings(
    NETWORK_ID,
    candidate_mappings,
    NETWORK_DIR / f'{OUTPUT_PREFIX}candidate_mappings.csv'
)
display(all_saved_mappings)

,network_id,vars_id,net_var_name,network_station_count,source_time_scale,canonical_variable,daily_aggregation,priority,source_family,mapping_status
0,76,1295,TEMP_MEAN,1,hourly,air_temperature,mean,1,network76_auto,needs_review
1,76,1297,avg_air_temp_pst1hr,1,hourly,air_temperature,mean,2,network76_auto,needs_review


## 7. Use the registry in the canonical-variable notebook

In [393]:
# Example
CANONICAL_VARIABLE = 'precipitation_amount'
candidate_rows = candidate_mappings[
    candidate_mappings['canonical_variable'] == CANONICAL_VARIABLE
]
candidate_ids = candidate_rows['vars_id'].tolist()
display(candidate_rows)

,vars_id,net_var_name,canonical_variable,daily_aggregation,priority,source_family,mapping_status,network_station_count,source_time_scale
